# Machine Learning Pipeline for Queen Bee Acoustic Monitoring

# Import Library & Setup Environment

In [2]:
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Run Model on: {DEVICE}")

Run Model on: cuda


In [3]:
RAW_AUDIO_DIR = 'dataset'
LABEL_CSV = 'dataset/state_labels.csv'

WINDOW_SEC = 5.0

df_labels = pd.read_csv(LABEL_CSV)
new_records = []

print("Finding all audio files in folder")
audio_files_map = {}
for root, dirs, files in os.walk(RAW_AUDIO_DIR):
    for file in files:
        if file.endswith('.wav') or file.endswith('.mp3'):
            base_name = os.path.splitext(file)[0]
            audio_files_map[base_name] = os.path.join(root, file)

print(f"Find all {len(audio_files_map)} File (Ready for matching with CSV)")

print("Start slicing audio file")
for _, row in tqdm(df_labels.iterrows(), total=len(df_labels)):
    sample_name = str(row['sample_name']).strip()
    label = str(row['label']).strip().lower()

    if label not in ['active', 'missing queen']: continue
    
    if sample_name not in audio_files_map:
        continue 
        
    file_path = audio_files_map[sample_name]
        
    try:
        info = torchaudio.info(file_path)
        sr = info.sample_rate
        total_frames = info.num_frames
        chunk_frames = int(sr * WINDOW_SEC)
        
        for start_frame in range(0, total_frames, chunk_frames):
            end_frame = start_frame + chunk_frames
            if end_frame > total_frames: break # ทิ้งเศษท้ายคลิป
            
            numeric_label = 0 if label == 'active' else 1
            new_records.append({
                'audio_path': file_path,          # ชี้ไปที่ไฟล์ 10 นาทีตัวเดิม
                'hive_id': sample_name.split('_')[0], 
                'start_frame': start_frame,       # พิกัดจุดเริ่มต้น
                'num_frames': chunk_frames,       # ความยาวที่ต้องโหลด
                'label': numeric_label
            })

    except Exception as e:
        pass

df_processed = pd.DataFrame(new_records)
df_processed.to_csv('virtual_hive_state_dataset.csv', index=False)
print(f"Finish slicing audio files, received {len(df_processed)} Files")

Finding all audio files in folder
Find all 576 File (Ready for matching with CSV)
Start slicing audio file


  0%|          | 0/573 [00:00<?, ?it/s]

Finish slicing audio files, received 67049 Files


In [4]:
TARGET_SR = 22050

class VirtualHiveStateDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        start_frame = int(row['start_frame'])
        num_frames = int(row['num_frames'])

        try:
            waveform, sr = torchaudio.load(row['audio_path'], frame_offset=start_frame, num_frames=num_frames)
            
            # Resample ถ้าเสียงไม่ได้มาในระดับ 22050 Hz
            if sr != TARGET_SR:
                waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=TARGET_SR)(waveform)
            
            # บังคับ Mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
                
        except Exception:
            # กันเหนียวเผื่อไฟล์อ่านไม่ได้ ให้ส่งเสียงเงียบไปแทน (เพื่อไม่ให้โค้ดพัง)
            waveform = torch.zeros((1, int(TARGET_SR * 5.0)))

        # --- Data Augmentation ---
        if self.augment:
            waveform = waveform * random.uniform(0.7, 1.2)
            if random.random() < 0.4:
                noise = torch.randn_like(waveform) * random.uniform(0.001, 0.01)
                waveform += noise
                
        # Normalize
        if waveform.abs().max() > 0:
            waveform = waveform / waveform.abs().max()
            
        label = torch.tensor(row['label'], dtype=torch.long)
        return waveform, label

df_all = pd.read_csv('virtual_hive_state_dataset.csv')

# Prevent Data Leakage: Split Train/Test by "Hive"
hives = df_all['hive_id'].unique()
train_hives, test_hives = train_test_split(hives, test_size=0.2, random_state=42)

df_train = df_all[df_all['hive_id'].isin(train_hives)]
df_test = df_all[df_all['hive_id'].isin(test_hives)]

# Build data loader
BATCH_SIZE = 64
train_loader = DataLoader(VirtualHiveStateDataset(df_train, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(VirtualHiveStateDataset(df_test, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Data is ready Train: {len(df_train)} | Test: {len(df_test)}")

Data is ready Train: 33778 | Test: 33271


In [7]:
class ModernHiveStateModel(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, 
            n_mels=64, 
            n_fft=1024,
            hop_length=512,
            f_min=50.0,   
            f_max=8000.0
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.resnet.fc.in_features, num_classes)
        )

    def forward(self, x):
        # x: (Batch, 1, Time)
        x = self.mel_spectrogram(x)
        x = self.amplitude_to_db(x)
        
        x = x.repeat(1, 3, 1, 1)
        return self.resnet(x)

model = ModernHiveStateModel(num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

print("Builted AI Model leaw :)")

Builted AI Model leaw :)


In [ ]:
EPOCHS = 20
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("Starting Training Modelll")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
    
    train_loss = train_loss / len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Test]"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_loss = val_loss / len(test_loader.dataset)
    val_acc = (correct / total) * 100

    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_hive_state_model.pth')
        print(f"Saved new model Highest Acc at {best_val_acc:.2f}%")

Starting Training Modelll


Epoch 1/20 [Train]:   0%|          | 0/528 [00:00<?, ?it/s]

Epoch 1/20 [Test]:   0%|          | 0/520 [00:00<?, ?it/s]

Epoch 1: Train Loss: 0.0739 | Val Loss: 1.5469 | Val Acc: 50.51%
Saved new model Highest Acc at 50.51%


Epoch 2/20 [Train]:   0%|          | 0/528 [00:00<?, ?it/s]

Epoch 2/20 [Test]:   0%|          | 0/520 [00:00<?, ?it/s]

Epoch 2: Train Loss: 0.0201 | Val Loss: 1.2372 | Val Acc: 64.49%
Saved new model Highest Acc at 64.49%


Epoch 3/20 [Train]:   0%|          | 0/528 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load('best_hive_state_model.pth'))
model.eval()

all_preds = []
all_labels = []

print("Last Evaluate")
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

target_names = ['Active (0)', 'Missing Queen (1)']
print("\n Classification Report:")
print(classification_report(all_labels, all_preds, target_names=target_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix: Hive State Detection', fontsize=14)
plt.ylabel('ความจริง (Actual)', fontsize=12)
plt.xlabel('AI ทำนาย (Predicted)', fontsize=12)
plt.show()